In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider
from astropy.table import Table
import astropy.units as u

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation
from Functions import *
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp as rpj
from astropy.convolution import convolve, convolve_fft
from scipy.ndimage import zoom, shift as ndi_shift
from photutils.centroids import centroid_quadratic
import time
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry

%matplotlib widget

def reproject_kernel_to_image(
    kernel_filepath,
    image_header,
    crop_size=None,
    normalize=True
):

    """
    Reproject PSF kernel onto image pixel scale/grid.
    """

    # =====================================================
    # LOAD KERNEL
    # =====================================================

    with fits.open(kernel_filepath) as hdul:

        kernel = hdul[0].data.astype(float)
        kernel_header = hdul[0].header

    kernel_wcs = WCS(kernel_header)

    # =====================================================
    # BUILD TARGET HEADER
    # =====================================================

    target_header = image_header.copy()

    # small output grid around center
    if crop_size is None:
        crop_size = 201

    target_header['NAXIS1'] = crop_size
    target_header['NAXIS2'] = crop_size

    target_header['CRPIX1'] = crop_size // 2 + 1
    target_header['CRPIX2'] = crop_size // 2 + 1

    target_header['CRVAL1'] = 0.0
    target_header['CRVAL2'] = 0.0

    # =====================================================
    # REPROJECT
    # =====================================================

    reproj_kernel, footprint = reproject_interp(
        (kernel, kernel_wcs),
        target_header,
        shape_out=(crop_size, crop_size)
    )

    # =====================================================
    # CLEAN
    # =====================================================

    reproj_kernel = np.nan_to_num(
        reproj_kernel,
        nan=0.0
    )

    # =====================================================
    # NORMALIZE
    # =====================================================

    if normalize:

        reproj_kernel /= np.sum(reproj_kernel)

    return reproj_kernel


class ImageScience:

    """
    Tools for continuum subtraction from filters.
    """


    def __init__(self):

        self.images = {}
        self.headers = {}
        self.files = {}
        self.wcs = {}

    def load_image(self, name, filename):

        """
        Load JWST FITS image properly with WCS.
        """

        hdul = fits.open(filename)

        self.files[name] = filename

        # --------------------------------------------------------
        # TRY SCI EXTENSION FIRST
        # --------------------------------------------------------

        try:

            data = hdul['SCI'].data.astype(float)

            header = hdul['SCI'].header

            wcs = WCS(hdul['SCI'].header, hdul)

        # --------------------------------------------------------
        # OTHERWISE USE PRIMARY
        # --------------------------------------------------------

        except:
            print('sci failed')
            data = hdul[0].data.astype(float)

            header = hdul[0].header

            wcs = WCS(hdul[0].header)

        self.images[name] = data
        self.headers[name] = header
        self.wcs[name] = wcs

        print(f'Loaded: {name}')

    def circular_mask(
        self,
        image_name,
        x_center,
        y_center,
        radius
    ):

        data = self.images[image_name].copy()

        yy, xx = np.indices(data.shape)

        r = np.sqrt(
            (xx - x_center)**2 +
            (yy - y_center)**2
        )

        mask = r <= radius

        data[mask] = np.nan

        self.images[image_name] = data

        print(f'Masked {image_name}')

    def check_alignment(
        self,
        image1,
        image2
    ):

        """
        Print basic alignment diagnostics.
        """

        data1 = self.images[image1]
        data2 = self.images[image2]

        print('----------------------------------')
        print('Alignment Diagnostics')
        print('----------------------------------')
        print(f'{image1} shape: {data1.shape}')
        print(f'{image2} shape: {data2.shape}')

        h1 = self.headers[image1]
        h2 = self.headers[image2]

        try:

            pix1 = abs(h1['CDELT1'])
            pix2 = abs(h2['CDELT1'])

            print(f'{image1} pixel scale: {pix1}')
            print(f'{image2} pixel scale: {pix2}')

        except:

            print('Could not determine CDELT1')

        print('----------------------------------')

    def align_images(
        self,
        reference_image,
        other_image,
        out_file=None
    ):

        if out_file is None:

            out_file = (
                self.files[other_image]
                .replace('.fits', '_aligned.fits')
            )

        print(
            f'Reprojecting {other_image} '
            f'onto {reference_image}'
        )

        # =====================================================
        # TARGET GEOMETRY
        # =====================================================

        target_header = self.headers[reference_image]

        target_shape = self.images[
            reference_image
        ].shape

        # =====================================================
        # REPROJECT
        # =====================================================

        reproj, footprint = reproject_interp(
            (
                self.images[other_image],
                self.wcs[other_image]
            ),
            self.wcs[reference_image],
            shape_out=target_shape
        )

        # =====================================================
        # SAVE
        # =====================================================

        hdu = fits.PrimaryHDU(
            data=reproj,
            header=target_header.copy()
        )

        hdu.writeto(
            out_file,
            overwrite=True
        )

        print(f'Saved aligned image:')
        print(out_file)

        # Reload into object
        self.load_image(
            f'{other_image}_aligned',
            out_file
        )
      
    def fft_convolve(self,
        image_name,
        kernel_filepath,
        normalize_kernel=True,
        preserve_nan=True,
        boundary='fill',
        fill_value=0.0,
        return_time=False
    ):
        """
        Convolve image using FFT convolution.

        Parameters
        ----------
        image : 2D ndarray
        kernel : 2D ndarray

        Returns
        -------
        convolved : ndarray
        elapsed_time : float (seconds)
        """
        
        

        # --------------------------------------------------------
        # START TIMER
        # --------------------------------------------------------

        t0 = time.perf_counter()
        kernel = reproject_kernel_to_image(kernel_filepath, self.headers[image_name], crop_size=None, normalize=True)

        # --------------------------------------------------------
        # COPY
        # --------------------------------------------------------

        image = self.images[image_name]

        # --------------------------------------------------------
        # CLEAN KERNEL
        # --------------------------------------------------------

        kernel = np.nan_to_num(kernel)

        # --------------------------------------------------------
        # NORMALIZE
        # --------------------------------------------------------

        if normalize_kernel:

            kernel /= np.sum(kernel)

        # --------------------------------------------------------
        # HANDLE NaNs
        # --------------------------------------------------------

        if preserve_nan:

            nan_mask = ~np.isfinite(image)

        image_filled = np.nan_to_num(image)

        # --------------------------------------------------------
        # FFT CONVOLUTION
        # --------------------------------------------------------

        convolved = convolve_fft(
            image_filled,
            kernel,
            boundary=boundary,
            fill_value=fill_value,
            normalize_kernel=False,
            preserve_nan=False,
            allow_huge=True
        )

        # --------------------------------------------------------
        # RESTORE NaNs
        # --------------------------------------------------------

        if preserve_nan:

            convolved[nan_mask] = np.nan

        # --------------------------------------------------------
        # END TIMER
        # --------------------------------------------------------

        elapsed = time.perf_counter() - t0

        self.images['fft_conv'] = convolved
        self.headers['fft_conv'] = self.headers[image_name]
        self.wcs['fft_conv'] = self.wcs[image_name]
        if return_time:
            print(f'fft convolution took {elapsed} seconds')

            return elapsed


    def convolve(self,
        image_name,
        kernel_filepath,
        normalize_kernel=True,
        preserve_nan=True,
        boundary='fill',
        fill_value=0.0,
        return_time=False
    ):

        """
        Convolve image using direct linear convolution.

        Parameters
        ----------
        image : 2D ndarray
        kernel : 2D ndarray

        Returns
        -------
        convolved : ndarray
        elapsed_time : float (seconds)
        """

        # --------------------------------------------------------
        # START TIMER
        # --------------------------------------------------------

        t0 = time.perf_counter()
        kernel = reproject_kernel_to_image(kernel_filepath, self.headers[image_name], crop_size=None, normalize=True)

        # --------------------------------------------------------
        # COPY
        # --------------------------------------------------------

        image = self.images[image_name]

        # --------------------------------------------------------
        # CLEAN KERNEL
        # --------------------------------------------------------

        kernel = np.nan_to_num(kernel)

        # --------------------------------------------------------
        # NORMALIZE
        # --------------------------------------------------------

        if normalize_kernel:

            kernel /= np.sum(kernel)

        # --------------------------------------------------------
        # HANDLE NaNs
        # --------------------------------------------------------

        if preserve_nan:

            nan_mask = ~np.isfinite(image)

        image_filled = np.nan_to_num(image)

        # --------------------------------------------------------
        # DIRECT CONVOLUTION
        # --------------------------------------------------------

        convolved = convolve(
            image_filled,
            kernel,
            boundary=boundary,
            fill_value=fill_value,
            normalize_kernel=False,
            preserve_nan=False
        )

        # --------------------------------------------------------
        # RESTORE NaNs
        # --------------------------------------------------------

        if preserve_nan:

            convolved[nan_mask] = np.nan

        # --------------------------------------------------------
        # END TIMER
        # --------------------------------------------------------

        elapsed = time.perf_counter() - t0

        print('----------------------------------')
        print('Direct Convolution Complete')
        print('----------------------------------')
        print(f'Time elapsed : {elapsed:.3f} sec')
        print('----------------------------------')

        self.images['cont_conv'] = convolved
        self.headers['cont_conv'] = self.headers[image_name]
        self.wcs['cont_conv'] = self.wcs[image_name]
        if return_time:
            print(f'convolution took {elapsed} seconds')
            return elapsed

    def continuum_subtract(
        self,
        f187_name,
        continuum_name,
        scale_factor,
        output_name='cont_subtracted'
    ):

        subtracted = (
            self.images[f187_name] -
            scale_factor * self.images[continuum_name]
        )

        self.images[output_name] = subtracted
        self.headers[output_name] = self.headers[
            f187_name
        ].copy()

        print(f'Created: {output_name}')

    def save_fits(
        self,
        image_name,
        output_file
    ):

        hdu = fits.PrimaryHDU(
            data=self.images[image_name],
            header=self.headers[image_name]
        )

        hdu.writeto(
            output_file,
            overwrite=True
        )

        print(f'Saved: {output_file}')

    def inspect_continuum_subtraction(
        self,
        feature_name,
        continuum_name,
        initial_scale=1.072,
        zoom_size=1000,
        mask_x=None,
        mask_y=None,
        mask_radius=None,
        show_all=False
    ):

        # -----------------------------------------------------
        # COPY DATA
        # -----------------------------------------------------

        feature = self.images[feature_name].copy()
        cont = self.images[continuum_name].copy()

        # -----------------------------------------------------
        # OPTIONAL MASK
        # -----------------------------------------------------

        if (
            mask_x is not None and
            mask_y is not None and
            mask_radius is not None
        ):

            yy, xx = np.indices(feature.shape)

            r = np.sqrt(
                (xx - mask_x)**2 +
                (yy - mask_y)**2
            )

            mask = r <= mask_radius

            feature[mask] = np.nan
            cont[mask] = np.nan

        # -----------------------------------------------------
        # CENTRAL CUTOUT
        # -----------------------------------------------------
        if zoom_size is not None:
            ny, nx = feature.shape

            x_center = nx // 2
            y_center = ny // 2

            x1 = x_center - zoom_size // 2
            x2 = x_center + zoom_size // 2

            y1 = y_center - zoom_size // 2
            y2 = y_center + zoom_size // 2

            feature_cut = feature[y1:y2, x1:x2]
            cont_cut = cont[y1:y2, x1:x2]
        else:
            feature_cut = feature
            cont_cut = cont

        # -----------------------------------------------------
        # INITIAL MODEL
        # -----------------------------------------------------

        continuum = initial_scale * cont_cut

        subtracted = feature_cut - continuum

        # -----------------------------------------------------
        # NORMALIZATION
        # -----------------------------------------------------
        if show_all:
            combined = np.concatenate([
                feature_cut[np.isfinite(feature_cut)].ravel(),
                continuum[np.isfinite(continuum)].ravel(),
                cont_cut[np.isfinite(cont_cut)].ravel()
            ])

            vmin = np.percentile(combined, 1)
            vmax = np.percentile(combined, 99.7)
            fig, axes = plt.subplots(
            2,
            2,
            figsize=(8, 8)
            )

            axes = axes.ravel()

        else:
            vmax = np.percentile(feature_cut[np.isfinite(feature_cut)].ravel(), 99.7)
            vmin = np.percentile(feature_cut[np.isfinite(feature_cut)].ravel(), 1)

            fig, axes = plt.subplots(figsize=(6, 6))



        sub_v = np.nanpercentile(
            np.abs(subtracted),
            99
        )

        # -----------------------------------------------------
        # FIGURE
        # -----------------------------------------------------


        plt.subplots_adjust(bottom=0.15)

        # -----------------------------------------------------
        # F187N
        # -----------------------------------------------------
        if show_all:
            im0 = axes[0].imshow(
                feature_cut,
                origin='lower',
                cmap='gray',
                vmin=vmin,
                vmax=vmax
            )

            axes[0].set_title('feature')
            axes[0].axis('off')

            # -----------------------------------------------------
            # CONTINUUM
            # -----------------------------------------------------

            im1 = axes[1].imshow(
                continuum,
                origin='lower',
                cmap='gray',
                vmin=vmin,
                vmax=vmax
            )

            title1 = axes[1].set_title(
                f'Continuum = {initial_scale:.5f}'
            )

            axes[1].axis('off')

            # -----------------------------------------------------
            # SUBTRACTED
            # -----------------------------------------------------

            im2 = axes[2].imshow(
                subtracted,
                origin='lower',
                cmap='RdBu_r',
                vmin=-sub_v,
                vmax=sub_v
            )

            title2 = axes[2].set_title(
                'feature - Continuum'
            )

            axes[2].axis('off')

            # -----------------------------------------------------
            # F150W
            # -----------------------------------------------------

            im3 = axes[3].imshow(
                cont_cut,
                origin='lower',
                cmap='gray',
                vmin=vmin,
                vmax=vmax
            )

            axes[3].set_title('Continuum Image')
            axes[3].axis('off')

            # -----------------------------------------------------
            # COLORBARS
            # -----------------------------------------------------

            plt.colorbar(
                im0,
                ax=axes[0],
                fraction=0.046
            )

            plt.colorbar(
                im1,
                ax=axes[1],
                fraction=0.046
            )

            plt.colorbar(
                im2,
                ax=axes[2],
                fraction=0.046
            )

            plt.colorbar(
                im3,
                ax=axes[3],
                fraction=0.046
            )

        else:
            im2 = axes.imshow(
                subtracted,
                origin='lower',
                cmap='RdBu_r',
                vmin=-sub_v,
                vmax=sub_v
            )

            title2 = axes.set_title(
                'feature - Continuum'
            )
            plt.colorbar(
                im2,
                ax=axes,
                fraction=0.046
            )

            axes.axis('off')

        # -----------------------------------------------------
        # SLIDER
        # -----------------------------------------------------

        ax_slider = plt.axes(
            [0.2, 0.05, 0.6, 0.03]
        )

        scale_slider = Slider(
            ax=ax_slider,
            label='Scale Factor',
            valmin=0.01,
            valmax=2,
            valinit=initial_scale,
            valstep=0.001
        )

        # -----------------------------------------------------
        # UPDATE
        # -----------------------------------------------------

        def update(val):

            scale = scale_slider.val

            continuum_new = scale * cont_cut

            subtracted_new = (
                feature_cut -
                continuum_new
            )
            if show_all:
                im1.set_data(continuum_new)
                title1.set_text(f'Continuum = {scale:.5f}')
            im2.set_data(subtracted_new)

            sub_v_new = np.nanpercentile(
                np.abs(subtracted_new),
                99
            )

            im2.set_clim(
                -sub_v_new,
                sub_v_new
            )


            title2.set_text(
                f'Median Residual = '
                f'{np.nanmedian(subtracted_new):.5e}'
            )

            fig.canvas.draw_idle()

        scale_slider.on_changed(update)

        plt.show()


    def get_background_subtracted_flux(
        self,
        image_name,
        loc,
        radius,
        background_annulus_thickness,
        buffer=0*u.arcsec
    ):

        """
        Robust circular aperture photometry with
        annulus background subtraction.

        Uses:
            - fractional pixel overlap
            - median background estimate per pixel
            - resistant to crowded fields

        Parameters
        ----------
        image_name : str

        loc : SkyCoord or [ra, dec]

        radius : astropy Quantity
            Source aperture radius.

        background_annulus_thickness : astropy Quantity
            Thickness of background annulus.

        buffer : astropy Quantity
            Gap between source aperture and annulus.

        Returns
        -------
        results : dict

            Contains:
                source_flux
                background_flux
                net_flux
                background_per_pixel
                source_area_pixels
                annulus_area_pixels
        """
        # LOAD FILE AND CHECK ARGUMENTS
        # =====================================================
        image = self.images[image_name]
        header = self.headers[image_name]
        wcs = self.wcs[image_name]

        if isinstance(loc, list):
            spatial_coords = SkyCoord(
                ra=loc[0] * u.deg,
                dec=loc[1] * u.deg
            )

        elif isinstance(loc, SkyCoord):
            spatial_coords = loc
        else:
            raise ValueError(
                'loc is not SkyCoord or [ra, dec]'
            )

        # UNIT HANDLING
        # =====================================================
        try:
            units = header['BUNIT']
        except:
            print('Units not found in header with key BUNIT, aperture photometry failed')
            return None
        if units == 'MJy/sr':
            original_units = u.MJy / u.sr
            image_quantity = (
                image * original_units
            ).to(
                u.W / (u.m**2 * u.Hz * u.sr)
            )
        elif units == "erg / (s cm2)":
            original_units = (
                u.erg / (u.s * u.cm**2)
            )
            pixel_area = (
                header['PIXAR_SR'] * u.sr
            )
            image_quantity = (
                (image * original_units) /
                pixel_area
            ).to(
                u.W / (u.m**2 * u.sr)
            )
        else:
            raise ValueError(
                f'Unsupported BUNIT: {units}'
            )

        try:
            pix_area = header['PIXAR_SR'] * u.sr
        except:
            print('Pixel area not found in header with key PIXAR_SR, aperture photometry failed')
            return None

        try:
            if header['CDELT2'] != header['CDELT1']:
                print('Pixels are not square!!!!!')
                return None
            pixel_scale_deg = abs(header['CDELT2'])
        except:
            print('Pixel size not found in header with key CDELT2, aperture photometry failed')
            return None
        
        source_radius_pixels = (radius.to_value(u.deg) / pixel_scale_deg)

        bg_inner_pixels = ((radius + buffer).to_value(u.deg) / pixel_scale_deg)

        bg_outer_pixels = ((radius + buffer + background_annulus_thickness).to_value(u.deg) / pixel_scale_deg)

        x, y = wcs.all_world2pix(
            spatial_coords.ra.deg,
            spatial_coords.dec.deg,
            0
        )

        # APPLY APERTURES
        # =====================================================

        source_aperture = CircularAperture(
            (x, y),
            r=source_radius_pixels
        )

        bg_annulus = CircularAnnulus(
            (x, y),
            r_in=bg_inner_pixels,
            r_out=bg_outer_pixels
        )

        source_flux = aperture_photometry(
            image_quantity,
            source_aperture,
            method='exact'
        )['aperture_sum'][0] * pix_area

        source_area_pixels = (
            source_aperture.area
        )

        annulus_mask = bg_annulus.to_mask(
            method='exact'
        )

        annulus_data = annulus_mask.multiply(
            image_quantity.value
        )

        annulus_weights = annulus_mask.data

        # VALID PIXELS
        # =====================================================

        valid = (
            np.isfinite(annulus_data) &
            (annulus_weights > 0)
        )

        annulus_values = annulus_data[valid]

        annulus_weights = annulus_weights[valid]

        # FRACTIONALLY WEIGHTED PIXELS
        # =====================================================

        # Recover intrinsic pixel values by dividing
        # weighted contributions by overlap fraction

        intrinsic_pixel_values = (
            annulus_values /
            annulus_weights
        )

        background_per_pixel = np.nanmedian(
            intrinsic_pixel_values
        ) * image_quantity.unit * pix_area

        annulus_area_pixels = np.sum(
            annulus_weights
        )

        # BACKGROUND INSIDE SOURCE
        # =====================================================

        background_flux = (
            background_per_pixel *
            source_area_pixels
        )

        net_flux = (
            source_flux -
            background_flux
        )

        return {
            'source_flux': source_flux,
            'background_flux': background_flux,
            'net_flux': net_flux,
            'background_per_pixel': background_per_pixel,
            'source_area_pixels': source_area_pixels,
            'annulus_area_pixels': annulus_area_pixels
        }

    def get_equivalent_width(
        feature_filter_file,
        continuum_filter_file,
        location,
        radius,
        annulus_inner_radius,
        annulus_outer_radius
    ):

        """
        Compute equivalent width using:
            - narrowband feature image
            - aligned/scaled continuum image

        Includes annular background subtraction.

        Parameters
        ----------
        feature_filter_file : str

        continuum_filter_file : str

        location : SkyCoord or [ra, dec] or (x, y)

        radius : float
            Aperture radius in pixels.

        annulus_inner_radius : float

        annulus_outer_radius : float

        Returns
        -------
        EW : astropy Quantity

        line_flux : astropy Quantity

        continuum_flux_density : astropy Quantity

        feature_flux : astropy Quantity

        continuum_flux : astropy Quantity
        """

        # =====================================================
        # BACKGROUND-SUBTRACTED PHOTOMETRY
        # =====================================================

        feature_flux, feature_bg = (
            get_background_subtracted_flux(
                feature_filter_file,
                location,
                radius,
                annulus_inner_radius,
                annulus_outer_radius
            )
        )

        continuum_flux, continuum_bg = (
            get_background_subtracted_flux(
                continuum_filter_file,
                location,
                radius,
                annulus_inner_radius,
                annulus_outer_radius
            )
        )

        # =====================================================
        # UNIT CHECK
        # =====================================================

        if feature_flux.unit != continuum_flux.unit:

            raise ValueError(
                'Feature and continuum images '
                'have different units.'
            )

        # =====================================================
        # FILTER INFO
        # =====================================================

        feature_filter = extract_filter_name(
            feature_filter_file
        )

        pivot = jwst_pivots[
            feature_filter
        ]

        wl, T = get_filter_data(
            feature_filter
        )

        # =====================================================
        # EFFECTIVE FILTER WIDTH
        # =====================================================

        bandwidth = (
            np.trapezoid(T, wl) /
            np.max(T)
        )

        # =====================================================
        # FNU -> FLAMBDA
        # =====================================================

        flam_feature = (
            feature_flux * c / pivot**2
        ).to(
            u.W / u.m**2 / u.m
        )

        flam_continuum = (
            continuum_flux * c / pivot**2
        ).to(
            u.W / u.m**2 / u.m
        )

        # =====================================================
        # TOTAL FLUX INSIDE FILTER
        # =====================================================

        feature_in_filter = (
            flam_feature * bandwidth
        )

        continuum_in_filter = (
            flam_continuum * bandwidth
        )

        # =====================================================
        # ISOLATED LINE FLUX
        # =====================================================

        line_flux = (
            feature_in_filter -
            continuum_in_filter
        )

        # =====================================================
        # EQUIVALENT WIDTH
        # =====================================================

        EW = (
            line_flux /
            flam_continuum
        ).to(u.Angstrom)

        return (
            EW,
            line_flux,
            flam_continuum,
            feature_flux,
            continuum_flux
        )


In [ ]:
pa_cont_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits'
pa_feature_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
ha_cont_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F689M_HST_WFC3_UVIS_IVM_drc.fits'
ha_feature_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits'
kernel_path = '/project/galaxies/tjuchau/data_files/JWST/PSFs/F150W_to_F187N.fits'
kernel = fits.getdata(kernel_path)
#initialize pa-a object
pcs = ImageScience()
#load images
pcs.load_image('cont', pa_cont_file)
pcs.load_image('feature', pa_feature_file)

pcs.align_images('feature', 'cont', out_file='cont_aligned')

pcs.fft_convolve('cont_aligned', kernel_path, return_time=True)
#pcs.convolve('cont_aligned', kernel_path, return_time=True)
pcs.continuum_subtract('feature', 'fft_conv', 1.072, output_name='cont_subtracted')
pcs.save_fits('cont_subtracted', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum_subtracted.fits')
pcs.save_fits('fft_conv', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum.fits')


'''
#initialize H-a object
hcs = ContinuumSubtraction()
#load images
hcs.load_image('cont', ha_cont_file)
hcs.load_image('feature', ha_feature_file)

#measure subpixel offset
shift = hcs.measure_shift('feature', 'cont', 8502, 5970, box_size=50, show_cutout=False)

#apply corrective shift
hcs.apply_shift('cont', shift, output_name='cont_alligned')
'''



In [ ]:
pcs = ImageScience()
pcs.load_image('cont', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum.fits')
pcs.load_image('cont_sub', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum_subtracted.fits')
pcs.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits')
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table = table[table['galaxy']=="M51"]
table

pcs.get_background_subtracted_flux(
        'f187',
        [table[0]['ra'],table[0]['dec']],
        1*u.arcsec,
        0.1*u.arcsec,
        buffer=0
    )


In [ ]:
# Interactive Paschen-alpha inspection
pcs.inspect_continuum_subtraction(
    feature_name='feature',
    continuum_name='fft_conv',
    initial_scale=1.072,
    zoom_size=500,
    mask_x=5200,
    mask_y=5200,
    mask_radius=250,
    #show_all=True
)

In [ ]:
# Interactive Paschen-alpha inspection without correction
pcs.inspect_continuum_subtraction(
    feature_name='feature',
    continuum_name='cont_aligned',
    initial_scale=1.072,
    zoom_size=500,
    mask_x=5200,
    mask_y=5200,
    mask_radius=250,
    #show_all=True
)

In [ ]:
# Interactive inspection
hcs.inspect_continuum_subtraction(
    feature_name='feature',
    continuum_name='cont_alligned',
    initial_scale=0.251,
    zoom_size=None,
    mask_x=None,
    mask_y=None,
    mask_radius=None
)

In [ ]:
pcs.continuum_subtract('feature', 'fft_conv', 1.072, output_name='cont_subtracted')
pcs.save_fits('cont_subtracted', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum_subtracted.fits')
pcs.save_fits('fft_conv', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum.fits')

#hcs.continuum_subtract('feature', 'cont_alligned', 0.251, output_name='cont_subtracted')
#hcs.save_fits('cont_subtracted', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_continuum_subtracted.fits')


In [ ]:
Ha = fits.open('/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_continuum_subtracted.fits')[0].data
Pa = fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum_subtracted.fits')[0].data
ha_v = np.nanpercentile(np.abs(Ha), 99)
pa_v = np.nanpercentile(np.abs(Pa), 99)
fig, axes = plt.subplots(1,2, figsize=(8, 4))

im0 = axes[0].imshow(
    Ha,
    origin='lower',
    cmap='RdBu_r',
    vmin=-ha_v,
    vmax=ha_v
)
plt.colorbar(
            im0,
            ax=axes[0],
            fraction=0.046
        )
axes[0].axis('off')

im1 = axes[1].imshow(
    Pa,
    origin='lower',
    cmap='RdBu_r',
    vmin=-pa_v,
    vmax=pa_v
)
plt.colorbar(
            im1,
            ax=axes[1],
            fraction=0.046
        )
axes[1].axis('off')


axes[0].set_title('Ha')
axes[1].set_title('Pa')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
import os
gal_names = ["ngc1433", 'ngc1512', 'ngc1672', "M51"]
galaxy_name = gal_names[3]


def continuum_subtract_f187(
    f150_file,
    f187_file,
    scale_factor,
    output_file=None
):
    """
    Continuum subtract a JWST F187N image using a scaled F150W image.

    Parameters
    ----------
    f150_file : str
        Path to the F150W FITS image.

    f187_file : str
        Path to the F187N FITS image.

    scale_factor : float
        Multiplicative scale factor applied to the F150W image.

    output_file : str, optional
        Output FITS filename.
        If None, a default filename is generated.

    Returns
    -------
    subtracted : ndarray
        Continuum-subtracted image array.
    """

    # ========================================================
    # LOAD DATA
    # ========================================================

    with fits.open(f150_file) as hdul150:
        f150_data = hdul150['SCI'].data.astype(float)
        f150_header = hdul150['SCI'].header

    with fits.open(f187_file) as hdul187:
        f187_data = hdul187['SCI'].data.astype(float)
        f187_header = hdul187['SCI'].header

    # ========================================================
    # CHECK SHAPES
    # ========================================================

    if f150_data.shape != f187_data.shape:
        raise ValueError(
            f'Image shapes do not match: '
            f'F150W {f150_data.shape} vs '
            f'F187N {f187_data.shape}'
        )

    # ========================================================
    # CONTINUUM SUBTRACTION
    # ========================================================

    continuum = scale_factor * f150_data

    subtracted = f187_data - continuum

    # ========================================================
    # OUTPUT FILENAME
    # ========================================================

    if output_file is None:

        base = os.path.splitext(os.path.basename(f187_file))[0]

        output_file = (
            f'{base}_contsub_scale{scale_factor:.5f}.fits'
        )

    # ========================================================
    # CREATE OUTPUT HEADER
    # ========================================================

    out_header = f187_header.copy()

    out_header['HISTORY'] = 'Continuum subtraction performed'
    out_header['CONTFILE'] = os.path.basename(f150_file)
    out_header['CONTSCL'] = scale_factor
    out_header['BUNIT'] = 'Continuum-subtracted flux'

    # ========================================================
    # WRITE FITS FILE
    # ========================================================

    hdu = fits.PrimaryHDU(
        data=subtracted,
        header=out_header
    )

    hdu.writeto(output_file, overwrite=True)

    print(f'Saved continuum-subtracted image:')
    print(f'    {output_file}')

    return subtracted

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from matplotlib.widgets import Slider

# ============================================================
# USER INPUTS
# ============================================================
galaxy_name = "M51"
if galaxy_name == "M51":
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_NGC5194_ACS_WFC_drc.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_cont_sub.fits'
elif galaxy_name == 'ngc1433':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_cont_sub.fits'

elif galaxy_name == 'ngc1512':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_cont_sub.fits'

elif galaxy_name == 'ngc1672':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1672/hlsp_phangs-hst_hst_wfc3-uvis_ngc1672mosaic_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_cont_sub.fits'

#f150_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F689M_HST_WFC3_UVIS_IVM_drc.fits'
#f187_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits'

try:
    f150 = fits.open(f150_file)['SCI'].data.astype(float)
    f187 = fits.open(f187_file)['SCI'].data.astype(float)
    f300 = fits.open(f300_file)['SCI'].data.astype(float)
except:
    f150 = fits.open(f150_file)[0].data.astype(float)
    f187 = fits.open(f187_file)[0].data.astype(float)
    f300 = fits.open(f300_file)[0].data.astype(float)
# Initial scale factor
initial_scale = 1.044

# Zoom region

# Set these manually after inspecting image size
x1, x2 = int(f150.shape[0]//2)-300, int(f150.shape[0]//2)+800
y1, y2 = int(f150.shape[1]//2)-300, int(f150.shape[1]//2)+800
mask_x = 5200
mask_y = 5200
mask_radius = 250

# ============================================================
# CREATE AGN MASK
# ============================================================

yy, xx = np.indices(f187.shape)

r = np.sqrt((xx - mask_x)**2 + (yy - mask_y)**2)

mask = r <= mask_radius

# Mask values with NaN
f187[mask] = np.nan
f150[mask] = np.nan

# ============================================================
# CUTOUT REGION
# ============================================================

f187_cut = f187[y1:y2, x1:x2]
f150_cut = f150[y1:y2, x1:x2]

# ============================================================
# INITIAL CONTINUUM MODEL
# ============================================================

continuum = initial_scale * f150_cut
subtracted = f187_cut - continuum

# ============================================================
# NORMALIZATION
# ============================================================

combined = np.concatenate([
    f187_cut[np.isfinite(f187_cut)].ravel(),
    continuum[np.isfinite(continuum)].ravel(),
    f150_cut[np.isfinite(f150_cut)].ravel()
])

vmin = np.percentile(combined, 1)
vmax = np.percentile(combined, 99.7)

sub_v = np.nanpercentile(np.abs(subtracted), 99)

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.ravel()
plt.subplots_adjust(bottom=0.15)

# ------------------------------------------------------------
# F187N
# ------------------------------------------------------------

im0 = axes[0].imshow(
    f187_cut,
    origin='lower',
    cmap='gray',
    vmin=vmin,
    vmax=vmax
)

axes[0].set_title('F187N')
axes[0].axis('off')

# ------------------------------------------------------------
# Continuum
# ------------------------------------------------------------

im1 = axes[1].imshow(
    continuum,
    origin='lower',
    cmap='gray',
    vmin=vmin,
    vmax=vmax
)

title1 = axes[1].set_title(f'Continuum = {initial_scale:.5f} × F150W')
axes[1].axis('off')

# ------------------------------------------------------------
# Subtracted
# ------------------------------------------------------------

im2 = axes[2].imshow(
    subtracted,
    origin='lower',
    cmap='RdBu_r',
    vmin=-sub_v,
    vmax=sub_v
)

title2 = axes[2].set_title('F187N - Continuum')
axes[2].axis('off')

# ------------------------------------------------------------
# f150
# ------------------------------------------------------------

im3 = axes[3].imshow(
    f150_cut,
    origin='lower',
    cmap='gray',
    vmin=-vmin,
    vmax=vmax
)

axes[3].set_title('F150W')
axes[3].axis('off')

# ============================================================
# COLORBARS
# ============================================================

plt.colorbar(im0, ax=axes[0], fraction=0.046)
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.colorbar(im3, ax=axes[3], fraction=0.046)

# ============================================================
# SLIDER
# ============================================================

ax_slider = plt.axes([0.2, 0.05, 0.6, 0.03])

scale_slider = Slider(
    ax=ax_slider,
    label='Scale Factor',
    valmin=0.01,
    valmax=2,
    valinit=initial_scale,
    valstep=0.001
)

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update(val):

    scale = scale_slider.val

    continuum_new = scale * f150_cut
    subtracted_new = f187_cut - continuum_new

    im1.set_data(continuum_new)
    im2.set_data(subtracted_new)

    sub_v_new = np.nanpercentile(np.abs(subtracted_new), 99)

    im2.set_clim(-sub_v_new, sub_v_new)

    title1.set_text(f'Continuum = {scale:.5f} × F150W')
    title2 = axes[2].set_title(f'F187N - Continuum <{np.nanmedian(subtracted_new)}>')

    fig.canvas.draw_idle()

scale_slider.on_changed(update)

plt.show()

In [ ]:
from astropy.table import Table
cont_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum.fits'
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table = table[table['galaxy'] == "M51"]
ages = []
ews = []
for row in table:
    loc = [row['ra'], row['dec']]
    radius = row['radius']*u.arcsec
    old_ew, f187_fnu, f150_fnu, f300_fnu = get_EW_using_filters(f187_file, [f150_file, f300_file], loc, radius)
    new_ew, *_ = get_equivalent_width(f187_file, cont_file, loc, radius)
    ages.append(row['best.sfh.age'])
    ews.append(new_ew)
    print(old_ew, new_ew)


In [ ]:
fig, ax = plt.subplots()
ax.scatter(ages, [i.value for i in ews])
ax.set_xscale('log')
fig.show()

In [ ]:
ews

In [ ]:
fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits')['SCI'].header

In [ ]:
print(fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits')['SCI'].header['CRVAL1'])
print(fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits')['SCI'].header['CRVAL1'])